# 04 — RAG Pipeline

## About

**Purpose:** Build a retrieval-augmented Q&A pipeline over the OIG LEIE exclusion records so users can ask natural-language questions about excluded providers.<br>
**Author:** Ganapathy K<br>
**Date:** 2026-05-15<br>
**Notes:** 8,482 LEIE records with valid NPI are converted to documents and indexed in a self-hosted Qdrant collection (Docker, port 6333). Embeddings run locally via `sentence-transformers`. LLM is Gemini 2.5 Flash via `GOOGLE_API_KEY_HEALTHCARE_PROVIDER_TERMINATION` loaded from `.env`. The Qdrant build and `ask()` cells call Gemini — re-run with care.<br>
**Description:** Loads the labelled NPPES dataset and the OIG LEIE CSV, filters LEIE to records with a valid NPI, converts each row into a natural-language document, embeds with `all-MiniLM-L6-v2` (384-dim), and indexes in a Qdrant collection named `leie_exclusions` using Cosine distance. A helper `ask()` retrieves top-3 documents and routes them with the question to Gemini, then displays the answer alongside its sources. The Qdrant collection persists in a Docker host-mounted volume (`qdrant_storage/`) so notebook 05 (LangGraph agent) can connect to the same collection without rebuilding.

### Change Control

| Date       | Version | Author      | Changes                                                                                |
|------------|---------|-------------|----------------------------------------------------------------------------------------|
| 2026-05-15 | 1.0     | Ganapathy K | Initial version (FAISS)                                                                |
| 2026-05-18 | 1.1     | Ganapathy K | Swap FAISS for self-hosted Qdrant (Docker); add Why-Qdrant rationale; revert auth to API key (ADC not configured on this machine) |

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Setup
### 1.1 Imports

In [2]:
import warnings
import os
import logging
from datetime import datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from google import genai

from langchain_community.docstore.document import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

warnings.filterwarnings("ignore")

D:\Data Science\Visual Studio Code\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.2 Configure logging

In [3]:
log_folder = Path("logs")
log_folder.mkdir(exist_ok=True)
log_filename = log_folder / f"run_{datetime.now().strftime('%Y-%m-%d')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler(log_filename, encoding='utf-8'),
        logging.StreamHandler(),
    ],
    force=True,
)
logger = logging.getLogger(__name__)

### 1.3 Config

In [4]:
processed_file_path = Path("D:/Data Science/Visual Studio Code/healthcare-provider-termination/data/processed/labelled_dataset.parquet")
leie_file_path = Path("D:/Data Science/Visual Studio Code/healthcare-provider-termination/data/raw/oig_leie_202602.csv")

qdrant_url = "http://localhost:6333"
qdrant_collection_name = "leie_exclusions"
embedding_dim = 384
distance_metric = Distance.COSINE

embedding_model_name = "all-MiniLM-L6-v2"
gemini_model_name = "gemini-2.5-flash"
retriever_k = 3

### 1.4 Authentication

In [5]:
load_dotenv()
google_api_key = os.environ["GOOGLE_API_KEY"]
logger.info("Gemini API key loaded from env")

2026-05-18 02:13:35,618 | INFO | Gemini API key loaded from env


## 2. Data Ingestion
### 2.1 Load Labelled Dataset

In [6]:
providers_raw_df = pd.read_parquet(processed_file_path)
providers_df = providers_raw_df.copy()
logger.info(f"Loaded labelled dataset — shape: {providers_df.shape}")

2026-05-18 02:13:37,228 | INFO | Loaded labelled dataset — shape: (500000, 331)


### 2.2 Load OIG LEIE

In [7]:
leie_raw_df = pd.read_csv(leie_file_path, encoding='latin-1', low_memory=False)
leie_df = leie_raw_df.copy()
logger.info(f"Loaded LEIE — shape: {leie_df.shape}")

2026-05-18 02:13:37,414 | INFO | Loaded LEIE — shape: (82749, 18)


## 3. Document Preparation
### 3.1 Filter LEIE to Valid NPIs

In [8]:
leie_valid_npi_df = leie_df[leie_df['NPI'] != 0].copy()

leie_rag_df = leie_valid_npi_df[['NPI', 'LASTNAME', 'FIRSTNAME', 'BUSNAME', 'SPECIALTY', 'STATE', 'EXCLTYPE', 'EXCLDATE', 'GENERAL']].copy()

logger.info(f"LEIE records with valid NPI: {leie_rag_df.shape[0]}")

2026-05-18 02:13:37,484 | INFO | LEIE records with valid NPI: 8482


### 3.2 Create Document Text

In [9]:
def create_document_text(row):
    name = row['BUSNAME'] if pd.notna(row['BUSNAME']) else f"{row['FIRSTNAME']} {row['LASTNAME']}"
    return (
        f"{name} is a {row['SPECIALTY']} in {row['STATE']} "
        f"who was excluded on {row['EXCLDATE']} "
        f"for {row['EXCLTYPE']} ({row['GENERAL']})."
    )

leie_rag_df['document_text'] = leie_rag_df.apply(create_document_text, axis=1)
logger.info(f"Sample document: {leie_rag_df['document_text'].iloc[0]}")

2026-05-18 02:13:37,626 | INFO | Sample document: 101 FIRST CARE PHARMACY INC is a PHARMACY in NY who was excluded on 20220320 for 1128b8 (OTHER BUSINESS).


### 3.3 Build LangChain Documents

In [10]:
documents = [
    Document(
        page_content=row['document_text'],
        metadata={
            'NPI': row['NPI'],
            'NAME': row['BUSNAME'] if pd.notna(row['BUSNAME']) else f"{row['FIRSTNAME']} {row['LASTNAME']}",
            'SPECIALTY': row['SPECIALTY'] if pd.notna(row['SPECIALTY']) else 'Unknown',
            'STATE': row['STATE'],
            'EXCLTYPE': row['EXCLTYPE'],
            'EXCLDATE': str(row['EXCLDATE']),
            'GENERAL': row['GENERAL'],
        }
    )
    for row in leie_rag_df.to_dict(orient='records')
]

logger.info(f"Built {len(documents)} documents")

2026-05-18 02:13:37,738 | INFO | Built 8482 documents


## 4. Vector Store

In [11]:
embedding_model = HuggingFaceEmbeddings(model_name=embedding_model_name)

qdrant_client = QdrantClient(url=qdrant_url)

if qdrant_client.collection_exists(qdrant_collection_name):
    qdrant_client.delete_collection(qdrant_collection_name)
    logger.info(f"Dropped existing collection '{qdrant_collection_name}' for a clean rebuild")

qdrant_client.create_collection(
    collection_name=qdrant_collection_name,
    vectors_config=VectorParams(size=embedding_dim, distance=distance_metric),
)
logger.info(f"Created collection '{qdrant_collection_name}' (dim={embedding_dim}, distance={distance_metric.value})")

vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name=qdrant_collection_name,
    embedding=embedding_model,
)
vector_store.add_documents(documents)

retriever = vector_store.as_retriever(search_kwargs={"k": retriever_k})

collection_info = qdrant_client.get_collection(qdrant_collection_name)
logger.info(f"Vector store built — {collection_info.points_count} vectors indexed in collection '{qdrant_collection_name}'")
logger.info(f"Retriever ready (k={retriever_k})")

2026-05-18 02:13:39,401 | INFO | No device provided, using cpu


2026-05-18 02:13:39,944 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-05-18 02:13:39,951 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"


2026-05-18 02:13:40,205 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


2026-05-18 02:13:40,205 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


...

2026-05-18 02:14:07,517 | INFO | HTTP Request: PUT http://localhost:6333/collections/leie_exclusions/points?wait=true "HTTP/1.1 200 OK"


2026-05-18 02:14:07,614 | INFO | HTTP Request: PUT http://localhost:6333/collections/leie_exclusions/points?wait=true "HTTP/1.1 200 OK"


2026-05-18 02:14:07,627 | INFO | HTTP Request: GET http://localhost:6333/collections/leie_exclusions "HTTP/1.1 200 OK"


2026-05-18 02:14:07,628 | INFO | Vector store built — 8482 vectors indexed in collection 'leie_exclusions'


2026-05-18 02:14:07,628 | INFO | Retriever ready (k=3)


## 5. LLM + RAG Query
### 5.1 Initialise Gemini Client

In [12]:
genai_client = genai.Client(api_key=google_api_key)
logger.info("Gemini client ready")

2026-05-18 02:14:08,433 | INFO | Gemini client ready


### 5.2 Define ask() Helper

In [13]:
def ask(query):
    retrieved_docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in retrieved_docs])
    prompt = f"Based on the following LEIE exclusion records:\n{context}\n\nAnswer this question: {query}"
    response = genai_client.models.generate_content(model=gemini_model_name, contents=prompt)

    print(f"Q: {query}")
    print(f"A: {response.text}")
    print("Sources:")
    for i, doc in enumerate(retrieved_docs):
        meta = doc.metadata
        print(f"  {i+1}. {meta['NAME']} | NPI: {meta['NPI']} | {meta['SPECIALTY']} | {meta['STATE']} | Excluded: {meta['EXCLDATE']} | Reason: {meta['GENERAL']}")
    print()

ask("Are there any excluded pharmacies in New York?")

2026-05-18 02:14:08,548 | INFO | HTTP Request: POST http://localhost:6333/collections/leie_exclusions/points/query "HTTP/1.1 200 OK"


2026-05-18 02:14:08,550 | INFO | AFC is enabled with max remote calls: 10.


2026-05-18 02:14:11,877 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Q: Are there any excluded pharmacies in New York?
A: Yes, based on the records provided, there are excluded pharmacies in New York.

The following are examples:
*   **NYC PHARMACY, INC** (PHARMACY in NY)
*   **NEW YORK PHARMACY, INC** (PHARMACY in NY)
*   **VITAL PHARMACIES, INC** (PHARMACY in NY)
Sources:
  1. NYC PHARMACY, INC | NPI: 1609912088 | PHARMACY | NY | Excluded: 20200319 | Reason: OTHER BUSINESS
  2. NEW YORK PHARMACY, INC | NPI: 1336221126 | PHARMACY | NY | Excluded: 20200319 | Reason: OTHER BUSINESS
  3. VITAL PHARMACIES, INC | NPI: 1295801389 | PHARMACY | NY | Excluded: 20200220 | Reason: OTHER BUSINESS



### 5.3 More Sample Queries

In [14]:
ask("Which individuals were excluded for fraud?")
ask("Are there any excluded providers in Texas?")

2026-05-18 02:14:11,987 | INFO | HTTP Request: POST http://localhost:6333/collections/leie_exclusions/points/query "HTTP/1.1 200 OK"


2026-05-18 02:14:11,988 | INFO | AFC is enabled with max remote calls: 10.


2026-05-18 02:14:14,945 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


2026-05-18 02:14:14,957 | INFO | HTTP Request: POST http://localhost:6333/collections/leie_exclusions/points/query "HTTP/1.1 200 OK"


2026-05-18 02:14:14,958 | INFO | AFC is enabled with max remote calls: 10.


Q: Which individuals were excluded for fraud?
A: All of the individuals listed were excluded for fraud:

*   **RUDOLPH WASHINGTON**
*   **MARITSA LEVONYAN**
*   **PHILLIP JONES**

The exclusion basis "1128a1" refers to mandatory exclusions due to convictions for offenses related to Medicare or State health care program fraud.
Sources:
  1. RUDOLPH WASHINGTON | NPI: 1952607236 | COMM MNTL HLTH CNTR | CA | Excluded: 20190320 | Reason: EMPLOYEE - PRIVATE S
  2. MARITSA LEVONYAN | NPI: 1932495025 | COMM MNTL HLTH CNTR | CA | Excluded: 20180920 | Reason: EMPLOYEE - PRIVATE S
  3. PHILLIP JONES | NPI: 1124447248 | NO KNOWN AFFILIATION | IN | Excluded: 20200120 | Reason: INDIVIDUAL (UNAFFILI



2026-05-18 02:14:17,278 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Q: Are there any excluded providers in Texas?
A: Yes, there are excluded providers in Texas.

The following providers in Texas are listed as excluded:
*   **TEXAS WELLNESS CLINIC** (excluded 20231221)
*   **AFFORDABLE PHARMACY, INC** (excluded 20211029)
*   **SOUTH TEXAS COMPREHENSIVE CANC** (excluded 20150920)
Sources:
  1. TEXAS WELLNESS CLINIC | NPI: 1063076651 | CLINIC | TX | Excluded: 20231221 | Reason: OTHER BUSINESS
  2. AFFORDABLE PHARMACY, INC | NPI: 1801231436 | PHARMACY | TX | Excluded: 20211029 | Reason: OTHER BUSINESS
  3. SOUTH TEXAS COMPREHENSIVE CANC | NPI: 1740236207 | HEMATOLOGY | TX | Excluded: 20150920 | Reason: PHYSICIAN PRACTICE (



## 6. Verify Persistence

In [15]:
verify_client = QdrantClient(url=qdrant_url)
persisted_info = verify_client.get_collection(qdrant_collection_name)
logger.info(
    f"Collection '{qdrant_collection_name}' persisted in Docker volume — "
    f"{persisted_info.points_count} vectors retained across client reconnects"
)

2026-05-18 02:14:17,575 | INFO | HTTP Request: GET http://localhost:6333/collections/leie_exclusions "HTTP/1.1 200 OK"


2026-05-18 02:14:17,576 | INFO | Collection 'leie_exclusions' persisted in Docker volume — 8482 vectors retained across client reconnects


## 7. Summary

**Pitch:** Built a RAG pipeline that lets users ask plain-English questions about OIG-excluded providers — local embeddings, **self-hosted Qdrant** retrieval, Gemini generation, with sources surfaced alongside every answer.

### Approach
- 8,482 LEIE records with valid NPI converted to natural-language documents with structured metadata (NPI, NAME, SPECIALTY, STATE, EXCLTYPE, EXCLDATE, GENERAL)
- Local embeddings via `all-MiniLM-L6-v2` (384-dim) — no API cost
- **Self-hosted Qdrant** (Docker, port 6333) holds the `leie_exclusions` collection of 8,482 vectors with Cosine distance; retriever returns top 3 per query
- Gemini 2.5 Flash via Application Default Credentials — `ask()` helper concatenates context + question and prints answer with sources

### Why Qdrant over FAISS
- **Production-grade:** real DB process, REST + gRPC APIs, restart-survivable via Docker volume — FAISS is an in-memory library, not a service
- **Data residency:** runs inside our own infra (and later inside the same GCP project as the LangGraph agent) — PHI never leaves the perimeter, no third-party BAA needed (Pinecone/Weaviate Cloud would require one)
- **Filterable metadata:** payload filters (e.g. `STATE == "TX"`) can be combined with vector search server-side — FAISS would need post-filtering in Python
- **Horizontally scalable:** ready for the full 82K-row LEIE once we expand beyond the valid-NPI subset

### Outputs
- Qdrant collection `leie_exclusions` persisted to `qdrant_storage/` (Docker host volume) — reloadable by notebook 05 without re-embedding

### Next iteration
- Update notebook 05 to load the Qdrant collection in place of `FAISS.load_local`
- Add a payload-filter example query (e.g. "excluded pharmacies in New York") that pushes the `STATE` filter into Qdrant rather than relying on the LLM to filter